# Cost-Aware Limit Order Book Forecasting — Kaggle Final Run

This notebook runs the final **h=40 (~10 s)** experiment using:

- `deeplob-colab-code` (attached Kaggle dataset)
- `btc-h40-cost` (attached Kaggle dataset)

Fixed setup:
- 100-snapshot input (~25 s)
- 40-snapshot horizon (~10 s)
- cost-aware short / flat / long target
- fee = 1.0 bp per side
- slippage = 0.5 bp per side
- confidence threshold = 0.50
- XGBoost vs DeepLOB vs Transformer
- identical portfolio backtest assumptions


In [ ]:
from pathlib import Path

CODE_DATASET_DIR = Path("/kaggle/input/deeplob-colab-code")
DATA_DATASET_DIR = Path("/kaggle/input/btc-h40-cost")

if not CODE_DATASET_DIR.exists():
    CODE_DATASET_DIR = Path("/kaggle/input/deeplob-colab-code")
if not DATA_DATASET_DIR.exists():
    DATA_DATASET_DIR = Path("/kaggle/input/btc-h40-cost")

WORK_DIR = Path("/kaggle/working/deeplob_run")
WRITABLE_PROJECT_DIR = Path("/kaggle/working/deeplob-cost-aware-starter")
OUTPUT_ROOT = Path("/kaggle/working/model_outputs/btc_h40_cost")
FINAL_DIR = Path("/kaggle/working/deeplob_final")

CONFIDENCE = 0.50
INITIAL_EQUITY = 100_000
NOTIONAL_PER_TRADE = 10_000
FEE_BPS = 1.0
SLIPPAGE_BPS = 0.5

DEEPLOB_TRAIN_SAMPLES = 1_000_000
TRANSFORMER_TRAIN_SAMPLES = 500_000
DEEPLOB_EPOCHS = 5
TRANSFORMER_EPOCHS = 5
BATCH_SIZE = 512

FORCE_RETRAIN = False

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("CODE_DATASET_DIR:", CODE_DATASET_DIR)
print("DATA_DATASET_DIR:", DATA_DATASET_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import tarfile
import zipfile

def find_project_root(root: Path):
    matches = list(root.rglob("pyproject.toml"))
    return matches[0].parent if matches else None

def find_processed_dir(root: Path):
    for p in root.rglob("btc_h40_cost"):
        if p.is_dir() and (p / "splits.npz").exists() and (p / "labels.npy").exists():
            return p
    for p in root.rglob("splits.npz"):
        parent = p.parent
        if (parent / "labels.npy").exists() and (parent / "metadata.json").exists():
            return parent
    return None

def find_archive(root: Path):
    supported = (".tar", ".tar.gz", ".tgz", ".zip")
    candidates = [
        p for p in root.rglob("*")
        if p.is_file() and p.name.lower().endswith(supported)
    ]
    return max(candidates, key=lambda p: p.stat().st_size) if candidates else None

def extract_archive(archive: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive} -> {destination}")
    name = archive.name.lower()
    if name.endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(destination)
    elif name.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(archive, "r:*") as tf:
            tf.extractall(destination)
    else:
        raise ValueError(f"Unsupported archive: {archive}")

SOURCE_PROJECT_DIR = find_project_root(CODE_DATASET_DIR)

if SOURCE_PROJECT_DIR is not None:
    print("Code dataset is already expanded.")
else:
    code_archive = find_archive(CODE_DATASET_DIR)
    if code_archive is None:
        raise FileNotFoundError(f"No project or supported archive under {CODE_DATASET_DIR}")
    code_extract = WORK_DIR / "code"
    extract_archive(code_archive, code_extract)
    SOURCE_PROJECT_DIR = find_project_root(code_extract)
    if SOURCE_PROJECT_DIR is None:
        raise FileNotFoundError("Could not find pyproject.toml after code extraction.")

DATA_DIR = find_processed_dir(DATA_DATASET_DIR)

if DATA_DIR is not None:
    print("Processed dataset is already expanded.")
else:
    data_archive = find_archive(DATA_DATASET_DIR)
    if data_archive is None:
        raise FileNotFoundError(f"No processed h=40 data or supported archive under {DATA_DATASET_DIR}")
    data_extract = WORK_DIR / "data"
    extract_archive(data_archive, data_extract)
    DATA_DIR = find_processed_dir(data_extract)
    if DATA_DIR is None:
        raise FileNotFoundError("Could not find processed h=40 data after extraction.")

print("\nSOURCE_PROJECT_DIR:", SOURCE_PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)

In [ ]:
import shutil
import subprocess
import sys
import os

def run(cmd, cwd=None, env=None):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )

if WRITABLE_PROJECT_DIR.exists():
    shutil.rmtree(WRITABLE_PROJECT_DIR)

shutil.copytree(SOURCE_PROJECT_DIR, WRITABLE_PROJECT_DIR)
PROJECT_DIR = WRITABLE_PROJECT_DIR

print("Writable PROJECT_DIR:", PROJECT_DIR)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-deps",
    "-e",
    str(PROJECT_DIR),
])

run([
    sys.executable,
    "-c",
    "import torch, xgboost, lob_project; "
    "print('torch:', torch.__version__); "
    "print('xgboost:', xgboost.__version__); "
    "print('project import: OK')"
])

In [ ]:
import torch
import subprocess

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Notebook Settings -> Accelerator.")

subprocess.run(["nvidia-smi"], check=False)

In [ ]:
import json
import numpy as np

metadata = json.loads((DATA_DIR / "metadata.json").read_text())
splits = np.load(DATA_DIR / "splits.npz")
labels = np.load(DATA_DIR / "labels.npy", mmap_mode="r")

print(json.dumps(metadata, indent=2)[:5000])

names = {0: "short", 1: "flat", 2: "long"}

for split_name in ["train", "validation", "test"]:
    idx = splits[split_name]
    y = labels[idx]
    values, counts = np.unique(y, return_counts=True)
    lookup = dict(zip(values.tolist(), counts.tolist()))

    print(f"\n{split_name}: {len(y):,} samples")
    for cls in [0, 1, 2]:
        n = int(lookup.get(cls, 0))
        print(f"  {names[cls]:5s}: {n:>10,} ({n / len(y):7.2%})")

In [ ]:
def run_portfolio(model_dir):
    script = PROJECT_DIR / "scripts/run_portfolio_backtest.py"

    help_result = subprocess.run(
        [sys.executable, str(script), "--help"],
        capture_output=True,
        text=True,
    )

    help_text = help_result.stdout + help_result.stderr

    cmd = [
        sys.executable,
        str(script),
        "--run-dir", str(model_dir),
        "--processed-dir", str(DATA_DIR),
        "--initial-equity", str(INITIAL_EQUITY),
        "--notional-per-trade", str(NOTIONAL_PER_TRADE),
        "--fee-bps", str(FEE_BPS),
        "--slippage-bps", str(SLIPPAGE_BPS),
    ]

    if "--confidence" in help_text:
        cmd += ["--confidence", str(CONFIDENCE)]
    elif "--min-confidence" in help_text:
        cmd += ["--min-confidence", str(CONFIDENCE)]
    elif "--confidence-threshold" in help_text:
        cmd += ["--confidence-threshold", str(CONFIDENCE)]
    else:
        print("No confidence CLI flag found; using script defaults.")

    run(cmd, cwd=PROJECT_DIR)

def find_metrics_file(model_dir):
    for path in [
        model_dir / "metrics.json",
        model_dir / "test_metrics.json",
    ]:
        if path.exists():
            return path
    raise FileNotFoundError(f"No metrics file found in {model_dir}")

## XGBoost

In [ ]:
xgb_dir = OUTPUT_ROOT / "xgboost"

if FORCE_RETRAIN and xgb_dir.exists():
    shutil.rmtree(xgb_dir)

xgb_artifacts = (
    list(xgb_dir.glob("*.joblib"))
    + list(xgb_dir.glob("*.ubj"))
    + list(xgb_dir.glob("model*.json"))
) if xgb_dir.exists() else []

if not xgb_artifacts:
    run([
        sys.executable,
        "scripts/train_model.py",
        "--model", "xgboost",
        "--processed-dir", str(DATA_DIR),
        "--output-dir", str(xgb_dir),
    ], cwd=PROJECT_DIR)
else:
    print("XGBoost model already exists; skipping training.")

run([
    sys.executable,
    "scripts/evaluate_model.py",
    "--run-dir", str(xgb_dir),
    "--processed-dir", str(DATA_DIR),
], cwd=PROJECT_DIR)

run_portfolio(xgb_dir)

## DeepLOB

In [ ]:
deeplob_dir = OUTPUT_ROOT / "deeplob"

if FORCE_RETRAIN and deeplob_dir.exists():
    shutil.rmtree(deeplob_dir)

deeplob_checkpoints = (
    list(deeplob_dir.glob("*.pt"))
    + list(deeplob_dir.glob("*.pth"))
) if deeplob_dir.exists() else []

if not deeplob_checkpoints:
    run([
        sys.executable,
        "scripts/train_model.py",
        "--model", "deeplob",
        "--processed-dir", str(DATA_DIR),
        "--output-dir", str(deeplob_dir),
        "--max-train-samples", str(DEEPLOB_TRAIN_SAMPLES),
        "--epochs", str(DEEPLOB_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
    ], cwd=PROJECT_DIR)
else:
    print("DeepLOB checkpoint already exists; skipping training.")

run([
    sys.executable,
    "scripts/evaluate_model.py",
    "--run-dir", str(deeplob_dir),
    "--processed-dir", str(DATA_DIR),
], cwd=PROJECT_DIR)

run_portfolio(deeplob_dir)

## Transformer

In [ ]:
transformer_dir = OUTPUT_ROOT / "transformer"

if FORCE_RETRAIN and transformer_dir.exists():
    shutil.rmtree(transformer_dir)

transformer_checkpoints = (
    list(transformer_dir.glob("*.pt"))
    + list(transformer_dir.glob("*.pth"))
) if transformer_dir.exists() else []

if not transformer_checkpoints:
    run([
        sys.executable,
        "scripts/train_model.py",
        "--model", "transformer",
        "--processed-dir", str(DATA_DIR),
        "--output-dir", str(transformer_dir),
        "--max-train-samples", str(TRANSFORMER_TRAIN_SAMPLES),
        "--epochs", str(TRANSFORMER_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
    ], cwd=PROJECT_DIR)
else:
    print("Transformer checkpoint already exists; skipping training.")

run([
    sys.executable,
    "scripts/evaluate_model.py",
    "--run-dir", str(transformer_dir),
    "--processed-dir", str(DATA_DIR),
], cwd=PROJECT_DIR)

run_portfolio(transformer_dir)

## Final comparison

In [ ]:
import pandas as pd

def first(d, *keys, default=float("nan")):
    for key in keys:
        if key in d:
            return d[key]
    return default

rows = []

for model in ["xgboost", "deeplob", "transformer"]:
    model_dir = OUTPUT_ROOT / model
    metrics_path = find_metrics_file(model_dir)
    print("Loading:", metrics_path)

    m = json.loads(metrics_path.read_text())

    backtest_path = model_dir / "portfolio_backtest" / "metrics.json"
    if not backtest_path.exists():
        raise FileNotFoundError(f"Backtest metrics not found: {backtest_path}")

    b = json.loads(backtest_path.read_text())

    rows.append({
        "model": model,
        "accuracy": first(m, "accuracy"),
        "macro_f1": first(m, "macro_f1"),
        "balanced_accuracy": first(m, "balanced_accuracy"),
        "mcc": first(m, "mcc"),
        "log_loss": first(m, "log_loss"),
        "ece": first(m, "ece", "ece_10"),
        "net_pnl": first(b, "net_pnl", "final_pnl"),
        "total_return": first(b, "total_return", "net_return"),
        "max_drawdown": first(b, "max_drawdown", "max_drawdown_fraction"),
        "trades": first(b, "trade_count", "n_trades", default=0),
        "win_rate": first(b, "win_rate"),
        "profit_factor": first(b, "profit_factor"),
    })

results = pd.DataFrame(rows).set_index("model")
results

In [ ]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(FINAL_DIR / "model_comparison.csv")
results.to_json(
    FINAL_DIR / "model_comparison.json",
    orient="index",
    indent=2,
)

print(results.to_string())
print("\nSaved:", FINAL_DIR / "model_comparison.csv")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(10, 5))
plotted = False

for model in ["xgboost", "deeplob", "transformer"]:
    path = OUTPUT_ROOT / model / "portfolio_backtest" / "equity_curve.csv"

    if not path.exists():
        print("Missing:", path)
        continue

    df = pd.read_csv(path)

    equity_col = next(
        (
            c for c in
            ["equity", "portfolio_equity", "account_value"]
            if c in df.columns
        ),
        None,
    )

    if equity_col is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        if numeric_cols:
            equity_col = numeric_cols[-1]

    if equity_col is not None:
        plt.plot(df[equity_col].to_numpy(), label=model)
        plotted = True

if plotted:
    plt.title("Out-of-Sample Portfolio Equity")
    plt.xlabel("Backtest step")
    plt.ylabel("Equity")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plot_path = FINAL_DIR / "equity_curves.png"
    plt.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("Saved:", plot_path)
else:
    print("No equity curves found.")

## Package Kaggle outputs

In [ ]:
import tarfile
import shutil

bundle = Path("/kaggle/working/deeplob_results")

if bundle.exists():
    shutil.rmtree(bundle)

bundle.mkdir(parents=True, exist_ok=True)

shutil.copytree(OUTPUT_ROOT, bundle / "btc_h40_cost")
shutil.copytree(FINAL_DIR, bundle / "final")

archive = Path("/kaggle/working/deeplob_results.tar.gz")

if archive.exists():
    archive.unlink()

with tarfile.open(archive, "w:gz") as tf:
    tf.add(bundle, arcname="deeplob_results")

print("Results archive:", archive)
print(f"Size: {archive.stat().st_size / 1e6:.1f} MB")
print("Save a Kaggle notebook version to preserve /kaggle/working outputs.")